In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim

import torchvision
from torchvision import datasets, transforms

from pathlib import Path

In [2]:
device = torch.device(
    'cuda' if torch.cuda.is_available() else 'cpu'
)
print(device)

cuda


In [3]:
train_transform = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(
        (0.4914, 0.4822, 0.4465),
        (0.2470, 0.2435, 0.2616)
    )
])

test_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(
        (0.4914, 0.4822, 0.4465),
        (0.2470, 0.2435, 0.2616)
    )
])

In [4]:
data_dir = Path(r"D:\projects\FoodLens\Data")

train_dataset = datasets.CIFAR10(
    root=data_dir,
    train=True,
    transform=train_transform,
    download=True
)

test_dataset = datasets.CIFAR10(
    root=data_dir,
    train=False,
    transform=test_transform,
    download=True
)

100.0%
C:\Users\kavya\AppData\Roaming\Python\Python313\site-packages\torchvision\datasets\cifar.py:83: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  entry = pickle.load(f, encoding="latin1")


In [5]:
from torch.utils.data import DataLoader

In [6]:
train_loader = DataLoader(
    train_dataset,
    batch_size=64,
    shuffle=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=64,
    shuffle=False
)

In [7]:
images, labels = next(iter(train_loader))

print("Images shape:", images.shape)
print("Labels shape:", labels.shape)

Images shape: torch.Size([64, 3, 32, 32])
Labels shape: torch.Size([64])


In [8]:
class CIFAR10(nn.Module):

    def __init__(self):
        super().__init__()

        self.features = nn.Sequential(

            nn.Conv2d(3, 32, 3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, 3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(64, 128, 3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(128, 256, 3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(),
            nn.MaxPool2d(2),
        )

        self.classifier = nn.Sequential(

            nn.AdaptiveAvgPool2d((1, 1)),
            nn.Flatten(),
            nn.Dropout(0.3),
            nn.Linear(256,128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128,10)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x

In [9]:
model = CIFAR10().to(device)

images, labels = next(iter(train_loader))

images = images.to(device)

output = model(images)

print(output.shape)

torch.Size([64, 10])


In [10]:
criterion = nn.CrossEntropyLoss()

optimizer = optim.Adam(
    model.parameters(),
    lr=0.001
)

epochs = 25

In [11]:
train_losses = []
train_accuracies = []

for epoch in range(epochs):

    model.train()

    running_loss = 0.0
    correct = 0
    total = 0

    for images, labels in train_loader:

        images = images.to(device)
        labels = labels.to(device)

        # Forward pass
        outputs = model(images)

        # Calculate loss
        loss = criterion(outputs, labels)

        # Clear old gradients
        optimizer.zero_grad()

        # Backpropagation
        loss.backward()

        # Update weights
        optimizer.step()

        # Track loss
        running_loss += loss.item()

        # Track accuracy
        predictions = outputs.argmax(dim=1)

        total += labels.size(0)
        correct += (predictions == labels).sum().item()

    epoch_loss = running_loss / len(train_loader)
    epoch_accuracy = 100 * correct / total

    train_losses.append(epoch_loss)
    train_accuracies.append(epoch_accuracy)

    print(
        f"Epoch [{epoch + 1}/{epochs}] "
        f"Loss: {epoch_loss:.4f} "
        f"Accuracy: {epoch_accuracy:.2f}%"
    )

Epoch [1/25] Loss: 1.4520 Accuracy: 46.65%
Epoch [2/25] Loss: 1.0876 Accuracy: 61.26%
Epoch [3/25] Loss: 0.9520 Accuracy: 66.40%
Epoch [4/25] Loss: 0.8669 Accuracy: 69.70%
Epoch [5/25] Loss: 0.8064 Accuracy: 71.91%
Epoch [6/25] Loss: 0.7492 Accuracy: 74.30%
Epoch [7/25] Loss: 0.7116 Accuracy: 75.44%
Epoch [8/25] Loss: 0.6801 Accuracy: 76.92%
Epoch [9/25] Loss: 0.6478 Accuracy: 77.90%
Epoch [10/25] Loss: 0.6186 Accuracy: 78.97%
Epoch [11/25] Loss: 0.5972 Accuracy: 79.60%
Epoch [12/25] Loss: 0.5771 Accuracy: 80.36%
Epoch [13/25] Loss: 0.5598 Accuracy: 80.99%
Epoch [14/25] Loss: 0.5440 Accuracy: 81.42%
Epoch [15/25] Loss: 0.5209 Accuracy: 82.34%
Epoch [16/25] Loss: 0.5077 Accuracy: 82.87%
Epoch [17/25] Loss: 0.4937 Accuracy: 83.27%
Epoch [18/25] Loss: 0.4863 Accuracy: 83.39%
Epoch [19/25] Loss: 0.4732 Accuracy: 83.96%
Epoch [20/25] Loss: 0.4660 Accuracy: 84.08%
Epoch [21/25] Loss: 0.4520 Accuracy: 84.76%
Epoch [22/25] Loss: 0.4452 Accuracy: 84.71%
Epoch [23/25] Loss: 0.4351 Accuracy: 85.1

In [12]:
correct = 0
total = 0
test_loss = 0.0

model.eval()

with torch.no_grad():

    for images, labels in test_loader:

        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)

        loss = criterion(outputs, labels)

        test_loss += loss.item()

        predictions = outputs.argmax(dim=1)

        total += labels.size(0)
        correct += (predictions == labels).sum().item()

test_loss = test_loss / len(test_loader)
test_accuracy = 100 * correct / total

print(f"Test Loss: {test_loss:.4f}")
print(f"Test Accuracy: {test_accuracy:.2f}%")

Test Loss: 0.4879
Test Accuracy: 83.44%


In [13]:
model.eval()

all_predictions = []
all_labels = []

with torch.no_grad():

    for images, labels in test_loader:

        images = images.to(device)

        outputs = model(images)

        predictions = outputs.argmax(dim=1)

        all_predictions.extend(predictions.cpu().numpy())
        all_labels.extend(labels.numpy())

all_predictions = np.array(all_predictions)
all_labels = np.array(all_labels)

print("Predictions:", all_predictions.shape)
print("Labels:", all_labels.shape)

Predictions: (10000,)
Labels: (10000,)


In [14]:
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(
    all_labels,
    all_predictions
)

print(cm)

[[917  11   8   9   7   2   4   3  21  18]
 [  9 948   1   2   0   2   2   0   5  31]
 [ 82   3 737  45  44  46  31   6   5   1]
 [ 23   2  42 722  43 102  36   9  15   6]
 [ 18   1  33  24 855  19  33  13   3   1]
 [ 16   4  18 140  33 752  13  19   4   1]
 [ 10   3  25  43  18  15 882   2   1   1]
 [ 35   2  24  46  43  55   7 773   3  12]
 [ 66  15   3  10   3   0   1   0 885  17]
 [ 30  75   3   6   2   1   1   2   7 873]]


In [17]:
model_path = r"D:\projects\CIFAR10_CNN\Models\cifar10_cnn.pth"

torch.save(
    model,
    model_path
)

print(f"Model saved to: {model_path}")

Model saved to: D:\projects\CIFAR10_CNN\Models\cifar10_cnn.pth


In [18]:
import json

class_names = train_dataset.classes

with open(
    r"D:\projects\CIFAR10_CNN\Models\class_names.json",
    "w"
) as f:
    json.dump(class_names, f)